# 02 — Cohort Definition

Takes the hourly time-series from notebook 01 and produces the final modeling cohort for the Sepsis-Associated AKI Early Warning System.

## Steps
1. **Inclusion criteria** — adults (Age ≥ 18) with ≥ 8 hours of ICU data.
2. **Baseline creatinine** — minimum serum creatinine at/before sepsis onset (fallback: first-24h minimum).
3. **Prevalent-AKI exclusion** — drop stays already meeting KDIGO Stage-1 criteria at sepsis onset, so the model learns *onset* rather than *persistence*.

Outputs `cohort_defined.parquet` and `baseline_creatinine.parquet` to `data/interim/`.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

ValueError: mount failed

In [ ]:
import os, sys
# point this at YOUR clone location if different
PROJ = '/content/drive/MyDrive/sentinel-poc/project_sentinel'
os.chdir(os.path.join(PROJ, 'notebooks'))   # so ../data/... paths resolve
sys.path.insert(0, PROJ)                      # so `from src...` works
print('cwd =', os.getcwd())

cwd = /content/drive/MyDrive/sentinel-poc/project_sentinel/notebooks


In [ ]:
import os
import sys
from pathlib import Path
import pandas as pd

# Add the project root to the path so we can import from src
sys.path.append(os.path.abspath('..'))
from src import data_loader

In [ ]:
interim = Path('../data/interim')
data_path = interim / 'hourly_cohort.parquet'

print(f"Loading hourly cohort from {data_path}...")
try:
    df = pd.read_parquet(data_path)
    print(f"Loaded {df['patient_id'].nunique()} ICU stays, {len(df):,} stay-hours.")
except FileNotFoundError:
    print(f"{data_path} not found. Run 01_data_acquisition.ipynb first.")
    df = pd.DataFrame()  # structural fallback

Loading hourly cohort from ../data/interim/hourly_cohort.parquet...
Loaded 140 ICU stays, 12,428 stay-hours.


In [ ]:
print("Applying inclusion criteria (Age >= 18, ICU stay >= 8h)...")
if not df.empty:
    df_included = data_loader.apply_inclusion_criteria(df, min_age=18, min_hours=8)
    print(f"Stays after inclusion criteria: {df_included['patient_id'].nunique()}")
else:
    df_included = df

Applying inclusion criteria (Age >= 18, ICU stay >= 8h)...
Stays after inclusion criteria: 136


In [ ]:
# Sepsis onset (t_sepsis) was already derived in notebook 01 and is carried in
# the hourly frame. Confirm it's present before defining the cohort.
if not df_included.empty:
    n_septic = df_included.loc[df_included['t_sepsis'].notna(), 'patient_id'].nunique()
    print(f"t_sepsis present for {n_septic} stays "
          f"(of {df_included['patient_id'].nunique()} included).")

t_sepsis present for 52 stays (of 136 included).


## Baseline Creatinine & Prevalent-AKI Exclusion

`define_cohort` computes a **baseline creatinine** per stay — the minimum serum creatinine at or before sepsis onset, falling back to the first-24h minimum, then the overall minimum.

It then **excludes stays with prevalent AKI** at sepsis onset (already meeting KDIGO Stage-1: creatinine ≥ baseline + 0.3 mg/dL or ≥ 1.5× baseline), so the model is trained to predict *new-onset* SA-AKI rather than recognise existing injury.

> The full MIMIC-IV uses pre-admission outpatient creatinine (7–365 days prior) for the baseline. The demo's short lookback means we approximate with the in-stay pre-sepsis minimum — a documented limitation to revisit on the full dataset.

In [ ]:
print("Defining final cohort (baseline creatinine + prevalent-AKI exclusion)...")
if not df_included.empty:
    cohort_df, baseline_df = data_loader.define_cohort(df_included)
    print(f"Final cohort: {cohort_df['patient_id'].nunique()} stays, {len(cohort_df):,} stay-hours.")

    # Persist for the feature / label engineering notebooks (03, 04)
    cohort_df.to_parquet(interim / 'cohort_defined.parquet', index=False)
    baseline_df.to_parquet(interim / 'baseline_creatinine.parquet', index=False)
    print(f"Saved → {interim / 'cohort_defined.parquet'}")
    print(f"Saved → {interim / 'baseline_creatinine.parquet'}")
else:
    cohort_df, baseline_df = df_included, pd.DataFrame()
    print("Empty input — nothing saved.")

Defining final cohort (baseline creatinine + prevalent-AKI exclusion)...
Final cohort: 136 stays, 12,413 stay-hours.
Saved → ../data/interim/cohort_defined.parquet
Saved → ../data/interim/baseline_creatinine.parquet


## Final Cohort Summary

`cohort_defined.parquet` now holds the hourly time-series for the modeling cohort — adult, ≥ 8h ICU stays with a baseline creatinine and prevalent AKI removed. `baseline_creatinine.parquet` holds one baseline value per stay.

**Next:** `03_feature_engineering.ipynb` builds rolling statistics, creatinine velocity, the urine-output features, and composites; `04_label_engineering.ipynb` applies the prospective KDIGO horizon labels (creatinine **and** urine output).

In [ ]:
%cd /content/drive/MyDrive/sentinel-poc
!git add -A
!git commit -m "run: 01_data_acquisition (clean)"
!git push

/content/drive/MyDrive/sentinel-poc
Author identity unknown

*** Please tell me who you are.

Run

  git config --global user.email "you@example.com"
  git config --global user.name "Your Name"

to set your account's default identity.
Omit --global to set the identity only in this repository.

fatal: unable to auto-detect email address (got 'root@ade6c3e6a964.(none)')
Everything up-to-date
